# Advanced Python Arithmetic Operators: A Step-by-Step Problem-Solving Tutorial

**A new collection of advanced problems, fully worked solutions, and executable experiments.**

This notebook continues the topic of the supplied *Arithmetic Operators* lesson, but uses **new problems**. Like the original lesson, we will introduce an operator, ask what Python should do, implement the behavior, run small experiments, and then explore a more subtle case.

**What you'll practice:** `__add__`, `__radd__`, `__sub__`, `__rsub__`, `__mul__`, `__rmul__`, `__matmul__`, `__rmatmul__`, `__truediv__`, `__rtruediv__`, `__pow__`, `__neg__`, `__abs__`, and augmented assignment. You'll also practice invariants, type dispatch, mathematical correctness, numerical pitfalls, and testing.

**Requirements:** Python 3.10+; only the standard library. Run the cells **in order**, preferably with **Kernel → Restart & Run All**. Every problem includes a statement, a staged solution, and executable assertions. This is a tutorial, not a collection of unconnected code answers.

## How to use this notebook

For each problem: (1) read the specification, (2) predict the behavior of the example, (3) read the reasoning in the next Markdown cell, (4) execute the implementation, and (5) inspect the tests. The final challenges invite you to change the design or add a method. All assertions are intended to pass; *expected* errors are captured explicitly so execution can continue.

Some design choices differ from the original `Vector` demonstration. In particular, we generally prefer **immutable value objects**, refuse `bool` as a numeric input, return `NotImplemented` for unsupported *operand types*, and raise a specific exception for invalid operations on supported types. These are deliberate extensions, not claims that the original lesson used those exact policies.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_EVEN
from fractions import Fraction
from math import isclose, isfinite, sqrt
from numbers import Real
from random import Random


def expect_error(exception_type, operation):
    """Confirm an example raises precisely the expected kind of exception."""
    try:
        operation()
    except exception_type as exc:
        print(f"Expected {type(exc).__name__}: {exc}")
        return
    raise AssertionError(f"Expected {exception_type.__name__}")


print("Tutorial setup complete.")

Tutorial setup complete.


---
# Problem 1 — Trace Python's arithmetic dispatch

**Difficulty:** Advanced language mechanics.

**Task:** Create two custom types that cooperate on `left + right`. The left operand should explicitly decline an unfamiliar operand; the right operand should then handle it. Next, investigate `+=` when `__iadd__` is absent, and the special ordering rule for a right-hand subclass.

Before looking at the solution, predict whether this statement is correct: *"Returning `NotImplemented` is the same as raising `NotImplementedError`."* It is not. One is a singleton result used by the binary-operation protocol; the other is an exception.

### Step 1 — Make the attempted methods observable

The standard operator dispatch is more subtle than simply calling `left.__add__(right)`: Python may try the reflected method on the other operand. `NotImplemented` says *try another applicable method*. It should be **returned**, not raised. Calling a dunder method directly exposes the sentinel, whereas using `+` gives Python a chance to finish dispatch.

In [2]:
dispatch_log = []

class Sender:
    def __add__(self, other):
        dispatch_log.append("Sender.__add__")
        return NotImplemented

class Receiver:
    def __radd__(self, other):
        dispatch_log.append("Receiver.__radd__")
        if isinstance(other, Sender):
            return "handled on the right"
        return NotImplemented

s, r = Sender(), Receiver()
assert s.__add__(r) is NotImplemented
assert s + r == "handled on the right"
print(dispatch_log)
assert dispatch_log[-2:] == ["Sender.__add__", "Receiver.__radd__"]

['Sender.__add__', 'Sender.__add__', 'Receiver.__radd__']


### Step 2 — Differentiate an unsupported type from a failed precondition

If both methods decline a pair of types, Python normally raises `TypeError` for `+`. That is different from, say, adding two matrices with incompatible dimensions: **both are valid matrix types**, but their shapes fail a mathematical precondition. For that situation, an informative `ValueError` is often preferable.

Do not write `raise NotImplemented`: it is not an exception class and it prevents correct cooperation.

In [3]:
expect_error(TypeError, lambda: Sender() + object())

class BrokenAdd:
    def __add__(self, other):
        raise NotImplementedError("This interrupts dispatch")

class WouldHandle:
    def __radd__(self, other):
        return "never reached"

expect_error(NotImplementedError, lambda: BrokenAdd() + WouldHandle())

Expected TypeError: unsupported operand type(s) for +: 'Sender' and 'object'
Expected NotImplementedError: This interrupts dispatch


### Step 3 — Augmented assignment need not preserve identity

When an applicable `__iadd__` exists, Python tries it. If it does not exist (or returns `NotImplemented`), augmented assignment falls back to ordinary addition and **rebinds the left-hand target**. To detect this, keep an alias to the original object; do not rely on printed memory addresses.

In [4]:
class ImmutableCounter:
    def __init__(self, number):
        self.number = number

    def __add__(self, other):
        if not isinstance(other, ImmutableCounter):
            return NotImplemented
        return ImmutableCounter(self.number + other.number)

    def __repr__(self):
        return f"ImmutableCounter({self.number})"

value = ImmutableCounter(3)
alias = value
value += ImmutableCounter(4)
assert value.number == 7
assert alias.number == 3
assert value is not alias
print("after +=:", value, "original alias:", alias)

after +=: ImmutableCounter(7) original alias: ImmutableCounter(3)


### Step 4 — An important subclass exception to left-first dispatch

For binary operators, if the right operand's type is a **proper subclass** of the left operand's type and provides a different reflected implementation, Python can give that reflected method priority. This allows a subtype to preserve its specialized semantics. The exact behavior relies on *different implementations*, not merely two instances of the same class.

In [5]:
priority_log = []

class BaseNumber:
    def __add__(self, other):
        priority_log.append("BaseNumber.__add__")
        return "base"

class SpecialNumber(BaseNumber):
    def __radd__(self, other):
        priority_log.append("SpecialNumber.__radd__")
        return "special"

assert BaseNumber() + SpecialNumber() == "special"
assert priority_log == ["SpecialNumber.__radd__"]
print(priority_log)

['SpecialNumber.__radd__']


**Takeaway:** An operator method should implement its own well-defined case, return `NotImplemented` for types it does not understand, and leave remaining dispatch to Python. Our later classes follow this rule. There is no reason to print from production operator methods merely to find out which one ran; the tracing code above is a teaching tool.

---
# Problem 2 — Build an immutable polynomial algebra

**Difficulty:** Advanced arithmetic protocol and algebraic invariants.

Represent a polynomial in increasing order of degree: `Polynomial(2, 3, 1)` denotes \(2+3x+x^2\). Support polynomial/scalar addition and subtraction in either order, polynomial multiplication, efficient nonnegative integer powers, evaluation via `p(x)`, and meaningful equality. Make `p += q` create a new polynomial instead of altering aliases.

**Constraints:** at least one coefficient, real numbers excluding booleans, finite inputs, and a canonical representation without unnecessary trailing zero coefficients. The zero polynomial must have one coefficient, `(0,)`.

### Step 1 — Decide what equality means before overloading `+`

`Polynomial(1, 2, 0, 0)` and `Polynomial(1, 2)` represent the same function. If we store both tuples literally, naive equality would disagree with the mathematics. So the constructor will normalize trailing zeros **once** and every operator will use the constructor to preserve that invariant.

We will avoid mutating polynomial objects: the same value may be shared safely by multiple names. Note that a `tuple` inside an object is not, by itself, a proof of full immutability; we also prevent attribute rebinding with a frozen dataclass.

In [6]:
def _clean_coefficients(values):
    values = tuple(values)
    if not values:
        raise ValueError("A polynomial needs at least one coefficient")
    for value in values:
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("Coefficients must be real numbers, not bool")
        if not isfinite(value):
            raise ValueError("Coefficients must be finite")
    while len(values) > 1 and values[-1] == 0:
        values = values[:-1]
    return values

assert _clean_coefficients((2, 3, 0, 0)) == (2, 3)
assert _clean_coefficients((0, 0, 0)) == (0,)
expect_error(TypeError, lambda: _clean_coefficients((True,)))
expect_error(ValueError, lambda: _clean_coefficients((1, float("inf"))))

Expected TypeError: Coefficients must be real numbers, not bool
Expected ValueError: Coefficients must be finite


### Step 2 — Work out the coefficient arithmetic independently

Addition aligns equal powers of \(x\); missing coefficients act like zero. Multiplication is a **convolution**: coefficient \(k\) is the sum of \(a_i b_j\) over every pair with \(i+j=k\). A degree-\(m\) polynomial times degree-\(n\) polynomial produces at most \(m+n+1\) coefficients.

Use simple reference functions first. Separating this reasoning from the operator methods makes it much easier to debug.

In [7]:
def add_coefficients(a, b):
    size = max(len(a), len(b))
    return tuple((a[i] if i < len(a) else 0) +
                 (b[i] if i < len(b) else 0) for i in range(size))


def multiply_coefficients(a, b):
    result = [0] * (len(a) + len(b) - 1)
    for i, left in enumerate(a):
        for j, right in enumerate(b):
            result[i + j] += left * right
    return tuple(result)

assert add_coefficients((1, 2), (3, 4, 5)) == (4, 6, 5)
assert multiply_coefficients((1, 1), (1, -1)) == (1, 0, -1)
print("(1+x)(1-x) coefficients:", multiply_coefficients((1, 1), (1, -1)))

(1+x)(1-x) coefficients: (1, 0, -1)


### Step 3 — Implement the complete value object

The scalar `3` is treated as the **constant polynomial** `Polynomial(3)`. Centralizing this conversion avoids duplicating every operation. For unrecognized types we return `NotImplemented`; for a negative or non-integer exponent, our documented power operation is undefined, so we raise `ValueError`/`TypeError` as appropriate.

Exponentiation by squaring reduces the number of polynomial multiplications from linear to logarithmic in the exponent (the polynomial multiplications themselves still have a cost). Evaluation uses Horner's rule to avoid repeatedly computing powers.

In [8]:
@dataclass(frozen=True)
class Polynomial:
    coefficients: tuple

    def __init__(self, *coefficients):
        object.__setattr__(self, "coefficients", _clean_coefficients(coefficients))

    @staticmethod
    def _convert(other):
        if isinstance(other, Polynomial):
            return other
        if isinstance(other, Real) and not isinstance(other, bool):
            return Polynomial(other)
        return NotImplemented

    @property
    def degree(self):
        return -1 if self.coefficients == (0,) else len(self.coefficients) - 1

    def __add__(self, other):
        rhs = self._convert(other)
        if rhs is NotImplemented:
            return NotImplemented
        return Polynomial(*add_coefficients(self.coefficients, rhs.coefficients))

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return Polynomial(*(-x for x in self.coefficients))

    def __sub__(self, other):
        rhs = self._convert(other)
        if rhs is NotImplemented:
            return NotImplemented
        return self + (-rhs)

    def __rsub__(self, other):
        lhs = self._convert(other)
        if lhs is NotImplemented:
            return NotImplemented
        return lhs - self

    def __mul__(self, other):
        rhs = self._convert(other)
        if rhs is NotImplemented:
            return NotImplemented
        return Polynomial(*multiply_coefficients(self.coefficients, rhs.coefficients))

    def __rmul__(self, other):
        return self * other

    def __pow__(self, exponent):
        if isinstance(exponent, bool) or not isinstance(exponent, int):
            raise TypeError("Exponent must be an integer, not bool")
        if exponent < 0:
            raise ValueError("Negative powers would require rational functions")
        result = Polynomial(1)
        factor = self
        while exponent:
            if exponent & 1:
                result = result * factor
            exponent >>= 1
            if exponent:
                factor = factor * factor
        return result

    def __call__(self, x):
        if isinstance(x, bool) or not isinstance(x, Real):
            raise TypeError("Evaluate at a real, non-boolean number")
        value = 0
        for coefficient in reversed(self.coefficients):
            value = value * x + coefficient
        return value

### Step 4 — Test ordinary, reflected, zero, and aliasing cases

A useful arithmetic test suite does more than check one example. In particular, subtraction is **not commutative**, so `2 - p` cannot simply delegate to `p - 2`. Likewise, a canonical zero prevents false equality failures after cancellation.

In [9]:
p = Polynomial(1, 2)          # 1 + 2x
q = Polynomial(-1, 0, 1)      # -1 + x^2
assert p + q == Polynomial(0, 2, 1)
assert 5 + p == Polynomial(6, 2)
assert 5 - p == Polynomial(4, -2)
assert p - 5 == Polynomial(-4, 2)
assert p * p == Polynomial(1, 4, 4)
assert (Polynomial(1, 1) ** 5).coefficients == (1, 5, 10, 10, 5, 1)
assert Polynomial(2, 3, 0) == Polynomial(2, 3)
assert (p - p) == Polynomial(0)
assert (p - p).degree == -1
assert p(3) == 7

original = p
p += Polynomial(10)
assert p == Polynomial(11, 2)
assert original == Polynomial(1, 2) and p is not original
print("Polynomial example:", p, "| original unchanged:", original)
expect_error(TypeError, lambda: Polynomial(1) + "invalid")
expect_error(ValueError, lambda: Polynomial(1, 1) ** -1)

Polynomial example: Polynomial(coefficients=(11, 2)) | original unchanged: Polynomial(coefficients=(1, 2))
Expected TypeError: unsupported operand type(s) for +: 'Polynomial' and 'str'
Expected ValueError: Negative powers would require rational functions


### Step 5 — Check algebraic laws, but state their limits

For **small integer coefficients**, equality is exact, so we can directly test distributivity: \(a(b+c)=ab+ac\). With arbitrary floating-point coefficients, different evaluation orders may round differently; use numerical tolerances where appropriate instead of assuming exact bit-for-bit equality.

In [10]:
a, b, c = Polynomial(1, 2), Polynomial(3, -1), Polynomial(2, 0, 1)
assert a * (b + c) == a * b + a * c
assert a + b == b + a
assert a * Polynomial(1) == a
assert a + Polynomial(0) == a
assert (a ** 0) == Polynomial(1)
print("Polynomial algebra checks: passed")

Polynomial algebra checks: passed


**Extension to try:** Implement a derivative method using \(\frac{d}{dx}\sum a_ix^i = \sum i a_i x^{i-1}\). Consider what the derivative of a constant polynomial must return and how you will preserve the canonical zero.

---
# Problem 3 — Dimension-aware quantities and safe arithmetic

**Difficulty:** Advanced domain modeling.

Design a value type `Quantity(magnitude, dimensions)` where dimensions is a three-tuple of integer exponents for **length, mass, and time**. Examples: meters `(1,0,0)`, seconds `(0,0,1)`, speed `(1,0,-1)`, and force `(1,1,-2)`.

Implement `+`, `-`, `*`, `/`, reflected division, integer powers, and equality. Addition must reject **physically incompatible dimensions**. Multiplication adds exponent tuples and division subtracts them. A plain scalar can multiply any quantity, but can only be added to a dimensionless quantity.

### Step 1 — Translate the physics into three tiny functions

The crucial distinction is that `2 meters + 3 seconds` is not a supported operation even though both objects are quantities. A descriptive `ValueError` communicates the violation. By contrast, `meters + "hello"` is an unknown operand *type*, so the operator should return `NotImplemented`.

The numeric system here intentionally uses floating-point magnitudes and integer dimension exponents. It is a compact teaching model, not a complete units library: it does **not** distinguish meters from centimeters when they share dimensions. Convert to common base units **before** constructing values.

In [11]:
DIMENSIONLESS = (0, 0, 0)
LENGTH = (1, 0, 0)
MASS = (0, 1, 0)
TIME = (0, 0, 1)


def combine_dims(left, right, sign=1):
    return tuple(a + sign * b for a, b in zip(left, right))


def raise_bad_number(number):
    if isinstance(number, bool) or not isinstance(number, Real):
        raise TypeError("Magnitude must be a real number, not bool")
    if not isfinite(number):
        raise ValueError("Magnitude must be finite")

assert combine_dims(LENGTH, TIME, -1) == (1, 0, -1)
assert combine_dims(MASS, LENGTH) == (1, 1, 0)

### Step 2 — Implement the immutable quantity

`__rtruediv__` is genuinely different from `__truediv__`: `1 / seconds` has dimensions of reciprocal time, whereas `seconds / 1` still has dimensions of time. We explicitly implement the reflected method instead of incorrectly forwarding it unchanged.

In [12]:
@dataclass(frozen=True)
class Quantity:
    magnitude: float
    dimensions: tuple[int, int, int] = DIMENSIONLESS

    def __post_init__(self):
        raise_bad_number(self.magnitude)
        if (not isinstance(self.dimensions, tuple) or len(self.dimensions) != 3
                or any(type(d) is not int for d in self.dimensions)):
            raise TypeError("Dimensions must be a tuple of exactly three integer exponents")

    def _addend(self, other):
        if isinstance(other, Quantity):
            return other
        if isinstance(other, Real) and not isinstance(other, bool):
            return Quantity(other)
        return NotImplemented

    def __add__(self, other):
        other = self._addend(other)
        if other is NotImplemented:
            return NotImplemented
        if self.dimensions != other.dimensions:
            raise ValueError("Cannot add quantities with incompatible dimensions")
        return Quantity(self.magnitude + other.magnitude, self.dimensions)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return Quantity(-self.magnitude, self.dimensions)

    def __sub__(self, other):
        other = self._addend(other)
        if other is NotImplemented:
            return NotImplemented
        return self + (-other)

    def __rsub__(self, other):
        other = self._addend(other)
        if other is NotImplemented:
            return NotImplemented
        return other - self

    def __mul__(self, other):
        if isinstance(other, Quantity):
            return Quantity(self.magnitude * other.magnitude,
                            combine_dims(self.dimensions, other.dimensions))
        if isinstance(other, Real) and not isinstance(other, bool):
            return Quantity(self.magnitude * other, self.dimensions)
        return NotImplemented

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        if isinstance(other, Quantity):
            return Quantity(self.magnitude / other.magnitude,
                            combine_dims(self.dimensions, other.dimensions, -1))
        if isinstance(other, Real) and not isinstance(other, bool):
            return Quantity(self.magnitude / other, self.dimensions)
        return NotImplemented

    def __rtruediv__(self, other):
        if isinstance(other, Real) and not isinstance(other, bool):
            return Quantity(other / self.magnitude, tuple(-x for x in self.dimensions))
        return NotImplemented

    def __pow__(self, exponent):
        if type(exponent) is not int:
            raise TypeError("Only integer powers are supported")
        return Quantity(self.magnitude ** exponent,
                        tuple(exponent * x for x in self.dimensions))

### Step 3 — Derive speed, acceleration, and force step by step

Use actual base-unit magnitudes: 100 meters in 20 seconds has speed 5 meters/second. Dividing speed by time yields acceleration; multiplying mass and acceleration yields force. Notice how the dimensions emerge from the operators, without special-case code for each physical quantity.

In [13]:
distance = Quantity(100, LENGTH)
elapsed = Quantity(20, TIME)
body_mass = Quantity(12, MASS)
speed = distance / elapsed
acceleration = speed / elapsed
force = body_mass * acceleration
assert speed == Quantity(5.0, (1, 0, -1))
assert acceleration == Quantity(0.25, (1, 0, -2))
assert force == Quantity(3.0, (1, 1, -2))
assert 2 * distance == Quantity(200, LENGTH)
assert 1 / elapsed == Quantity(0.05, (0, 0, -1))
assert elapsed ** 2 == Quantity(400, (0, 0, 2))
print("speed:", speed, "| force:", force)

speed: Quantity(magnitude=5.0, dimensions=(1, 0, -1)) | force: Quantity(magnitude=3.0, dimensions=(1, 1, -2))


### Step 4 — Probe errors and dimensionless arithmetic

For a dimensionless result, adding a bare real is meaningful in our chosen model. For a dimensional result, the same addition is rejected. Division by zero should be left to Python's `ZeroDivisionError`; do not silently return infinity.

In [14]:
ratio = Quantity(8, LENGTH) / Quantity(2, LENGTH)
assert ratio.dimensions == DIMENSIONLESS
assert ratio + 3 == Quantity(7.0)
assert 3 + ratio == Quantity(7.0)
expect_error(ValueError, lambda: distance + elapsed)
expect_error(ValueError, lambda: 3 + distance)
expect_error(TypeError, lambda: distance + "meter")
expect_error(ZeroDivisionError, lambda: distance / Quantity(0, TIME))
expect_error(TypeError, lambda: Quantity(1, (True, 0, 0)))

Expected ValueError: Cannot add quantities with incompatible dimensions
Expected ValueError: Cannot add quantities with incompatible dimensions
Expected TypeError: unsupported operand type(s) for +: 'Quantity' and 'str'
Expected ZeroDivisionError: division by zero
Expected TypeError: Dimensions must be a tuple of exactly three integer exponents


**Design discussion:** A real engineering units package needs named units, conversion factors, offset temperatures, and policies for uncertainty. The arithmetic protocol is the interface, not a substitute for a correct dimensional model.

---
# Problem 4 — Why points and displacement vectors need different operators

**Difficulty:** Type-directed operator design and affine geometry.

Implement a 2D `Point` and a 2D `Displacement` with these rules:

- Point − Point → Displacement.
- Point + Displacement → Point; Displacement + Point → Point.
- Displacement ± Displacement → Displacement; scalar × Displacement → Displacement.
- Point + Point is **not defined**: adding two locations has no coordinate-independent meaning in affine geometry.

Also demonstrate that `point += displacement` can create a new point without mutating an alias.

### Step 1 — Encode the mathematical rules in methods, not in caller conventions

We could reuse one class for every coordinate pair, but then `point + point` would become accidentally legal. Two types make illegal states harder to express. Immutable dataclasses will also give us structural equality and useful debug representations.

The displacement's `__add__` declines a point. The point's `__radd__` handles the reverse spelling `displacement + point`. This is a real use for the reflected addition protocol.

In [15]:
@dataclass(frozen=True)
class Displacement:
    dx: float
    dy: float

    def __add__(self, other):
        if not isinstance(other, Displacement):
            return NotImplemented
        return Displacement(self.dx + other.dx, self.dy + other.dy)

    def __neg__(self):
        return Displacement(-self.dx, -self.dy)

    def __sub__(self, other):
        if not isinstance(other, Displacement):
            return NotImplemented
        return self + (-other)

    def __mul__(self, other):
        if isinstance(other, Real) and not isinstance(other, bool):
            return Displacement(self.dx * other, self.dy * other)
        return NotImplemented

    def __rmul__(self, other):
        return self * other

    def __abs__(self):
        return sqrt(self.dx ** 2 + self.dy ** 2)


@dataclass(frozen=True)
class Point:
    x: float
    y: float

    def __add__(self, other):
        if isinstance(other, Displacement):
            return Point(self.x + other.dx, self.y + other.dy)
        return NotImplemented

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        if isinstance(other, Point):
            return Displacement(self.x - other.x, self.y - other.y)
        if isinstance(other, Displacement):
            return self + (-other)
        return NotImplemented

### Step 2 — Test the complete operator table

Keep one set of examples for valid operations and another for invalid operations. This gives a reviewer a concrete answer to the question, *"Which combinations does the API promise to support?"*

In [16]:
origin = Point(0, 0)
target = Point(3, 4)
move = Displacement(3, 4)
assert target - origin == move
assert origin + move == target
assert move + origin == target
assert target - move == origin
assert move + Displacement(-3, -4) == Displacement(0, 0)
assert 2 * move == Displacement(6, 8)
assert abs(move) == 5
expect_error(TypeError, lambda: origin + target)
expect_error(TypeError, lambda: origin * 2)
expect_error(TypeError, lambda: move - origin)
print("Coordinate rules: passed")

Expected TypeError: unsupported operand type(s) for +: 'Point' and 'Point'
Expected TypeError: unsupported operand type(s) for *: 'Point' and 'int'
Expected TypeError: unsupported operand type(s) for -: 'Displacement' and 'Point'
Coordinate rules: passed


### Step 3 — Explain the effect of `+=` on aliases

Since `Point` does not define `__iadd__`, `p += move` uses its ordinary addition protocol and reassigns the name `p` to a **new** point. Anyone holding the original alias continues to see the original location. This is often the safer behavior for a geometric value object.

In [17]:
position = Point(1, 1)
saved_position = position
position += Displacement(10, -2)
assert position == Point(11, -1)
assert saved_position == Point(1, 1)
assert position is not saved_position
print("Moved:", position, "| snapshot:", saved_position)

Moved: Point(x=11, y=-1) | snapshot: Point(x=1, y=1)


**Extension to try:** Define a `Path` storing a tuple of points. What should `path + displacement` mean? Would `path + path` have an equally natural meaning? Operator overloading should support a *clear domain interpretation*, not merely whatever implementation is convenient.

---
# Problem 5 — Sparse vectors with a fast dot product

**Difficulty:** Data invariants, mixed operator meanings, and algorithmic cost.

For a vector with a million coordinates but only five nonzero entries, storing a million-element tuple wastes space. Store an explicit dimension plus only nonzero `(index, value)` pairs. Support vector addition, subtraction, scalar multiplication (both orders), dot product using `@`, and Euclidean magnitude via `abs`.

**Decision:** Here `*` means *only scalar multiplication*; `@` means vector dot product. This follows a different, equally deliberate convention from the supplied lesson, in which `*` was also used for dot products.

### Step 1 — Define a canonical sparse representation

Every stored index must be in range and unique; zero entries are dropped and indices are sorted. Returning an immutable tuple of entries prevents callers from silently violating the invariant. The `dimension` must stay explicit: the all-zero vectors of dimensions 3 and 1,000,000 are not interchangeable.

In [18]:
@dataclass(frozen=True)
class SparseVector:
    dimension: int
    entries: tuple[tuple[int, float], ...]

    def __init__(self, dimension, entries=None):
        if type(dimension) is not int or dimension < 1:
            raise ValueError("Dimension must be a positive integer")
        data = {} if entries is None else dict(entries)
        for index, value in data.items():
            if type(index) is not int:
                raise TypeError("Sparse index must be an integer, not bool")
            if not 0 <= index < dimension:
                raise ValueError("Sparse index is outside the dimension")
            raise_bad_number(value)
        canonical = tuple(sorted((i, v) for i, v in data.items() if v != 0))
        object.__setattr__(self, "dimension", dimension)
        object.__setattr__(self, "entries", canonical)

    def as_dict(self):
        return dict(self.entries)

    def __add__(self, other):
        if not isinstance(other, SparseVector):
            return NotImplemented
        if self.dimension != other.dimension:
            raise ValueError("Sparse vector dimensions differ")
        result = self.as_dict()
        for index, value in other.entries:
            result[index] = result.get(index, 0) + value
        return SparseVector(self.dimension, result)

    def __neg__(self):
        return SparseVector(self.dimension, ((i, -v) for i, v in self.entries))

    def __sub__(self, other):
        if not isinstance(other, SparseVector):
            return NotImplemented
        return self + (-other)

    def __mul__(self, other):
        if not isinstance(other, Real) or isinstance(other, bool):
            return NotImplemented
        return SparseVector(self.dimension, ((i, v * other) for i, v in self.entries))

    def __rmul__(self, other):
        return self * other

    def __matmul__(self, other):
        if not isinstance(other, SparseVector):
            return NotImplemented
        if self.dimension != other.dimension:
            raise ValueError("Dot product needs equal dimensions")
        right = other.as_dict()
        return sum(value * right.get(index, 0) for index, value in self.entries)

    def __abs__(self):
        return sqrt(self @ self)

### Step 2 — Check that cancellation does not leave ghost entries

If a coordinate sums to zero, the constructor eliminates it. This is important for equality and for maintaining performance after repeated operations. The addition implementation traverses the stored entries, not the full index range.

In [19]:
sparse_a = SparseVector(1_000_000, {5: 3, 999_999: 4})
sparse_b = SparseVector(1_000_000, {5: -3, 8: 7})
combined = sparse_a + sparse_b
assert combined.entries == ((8, 7), (999_999, 4))
assert sparse_a @ sparse_b == -9
assert abs(sparse_a) == 5
assert (2 * sparse_a).entries == ((5, 6), (999_999, 8))
assert (sparse_a * 0).entries == ()
assert (sparse_a - sparse_a).entries == ()
print("Two nonzero coordinates out of", combined.dimension, "total:", combined.entries)

Two nonzero coordinates out of 1000000 total: ((8, 7), (999999, 4))


### Step 3 — Guard against silent `zip` truncation and invalid operands

A common beginner mistake is to call `zip` on two coordinate collections without checking dimensions. `zip` silently stops at the shorter input. Validate the dimension **before** doing any pairwise work, and do not let a vector be accidentally treated as a scalar.

In [20]:
expect_error(ValueError, lambda: sparse_a + SparseVector(2, {1: 1}))
expect_error(ValueError, lambda: sparse_a @ SparseVector(2, {1: 1}))
expect_error(TypeError, lambda: sparse_a * sparse_b)
expect_error(ValueError, lambda: SparseVector(3, {3: 1}))
expect_error(TypeError, lambda: SparseVector(3, {True: 2}))

Expected ValueError: Sparse vector dimensions differ
Expected ValueError: Dot product needs equal dimensions
Expected TypeError: unsupported operand type(s) for *: 'SparseVector' and 'SparseVector'
Expected ValueError: Sparse index is outside the dimension
Expected TypeError: Sparse index must be an integer, not bool


### Step 4 — Discuss cost realistically

With \(k\) stored elements in one vector and \(m\) in the other, dict-based addition takes expected \(O(k+m)\) work, plus sorting the output pairs; dot product takes expected \(O(k+m)\) work because we build a lookup for the right input. These are *expected* dictionary costs, not mathematical worst-case guarantees. For many repeated dot products against the same vector, cache a safe immutable lookup or use an alternative representation.

**Extension to try:** Implement `__getitem__` for random access while maintaining the representation invariant. What is the time/space trade-off between a dictionary stored internally and our sorted tuple?

---
# Problem 6 — Matrix multiplication and mixed `@` dispatch

**Difficulty:** Shape checking, matrix algebra, and interoperability.

Implement a dense `Matrix` type and a compatible dense `RowVector` type. Support:

- Matrix + Matrix (same shape).
- Matrix × scalar using `*` (both orders).
- Matrix @ Matrix (inner dimensions must match).
- Matrix @ RowVector (matrix times a column interpreted from its components).
- RowVector @ Matrix (a row vector times a matrix), **using the matrix's reflected `__rmatmul__`**.

Although both vector operations return a `RowVector` container, their orientations in the formulas differ. The purpose is to learn operator dispatch and shape contracts, not to claim vectors inherently have one orientation.

### Step 1 — Validate the shape once

A matrix must have at least one row and one column, all rows the same length, and finite real entries. Store rows as tuples to discourage accidental mutation. We will keep `RowVector.__matmul__` intentionally conservative and let `Matrix.__rmatmul__` implement the row-vector × matrix case.

In [21]:
@dataclass(frozen=True)
class RowVector:
    values: tuple

    def __init__(self, *values):
        if not values:
            raise ValueError("A vector must have at least one component")
        for value in values:
            raise_bad_number(value)
        object.__setattr__(self, "values", tuple(values))

    def __matmul__(self, other):
        return NotImplemented


@dataclass(frozen=True)
class Matrix:
    rows: tuple[tuple, ...]

    def __init__(self, rows):
        rows = tuple(tuple(row) for row in rows)
        if not rows or not rows[0] or any(len(row) != len(rows[0]) for row in rows):
            raise ValueError("Expected a nonempty rectangular matrix")
        for row in rows:
            for value in row:
                raise_bad_number(value)
        object.__setattr__(self, "rows", rows)

    @property
    def shape(self):
        return (len(self.rows), len(self.rows[0]))

    @property
    def T(self):
        return Matrix(zip(*self.rows))

    def __add__(self, other):
        if not isinstance(other, Matrix):
            return NotImplemented
        if self.shape != other.shape:
            raise ValueError(f"Cannot add shapes {self.shape} and {other.shape}")
        return Matrix((tuple(a + b for a, b in zip(ra, rb))
                       for ra, rb in zip(self.rows, other.rows)))

    def __mul__(self, scalar):
        if not isinstance(scalar, Real) or isinstance(scalar, bool):
            return NotImplemented
        return Matrix((tuple(scalar * x for x in row) for row in self.rows))

    def __rmul__(self, scalar):
        return self * scalar

    def __matmul__(self, other):
        if isinstance(other, Matrix):
            if self.shape[1] != other.shape[0]:
                raise ValueError(f"Cannot multiply shapes {self.shape} and {other.shape}")
            other_columns = tuple(zip(*other.rows))
            return Matrix((tuple(sum(x * y for x, y in zip(row, column))
                                 for column in other_columns) for row in self.rows))
        if isinstance(other, RowVector):
            if self.shape[1] != len(other.values):
                raise ValueError("Matrix × column-vector shape mismatch")
            return RowVector(*(sum(x * y for x, y in zip(row, other.values))
                               for row in self.rows))
        return NotImplemented

    def __rmatmul__(self, other):
        if not isinstance(other, RowVector):
            return NotImplemented
        if len(other.values) != self.shape[0]:
            raise ValueError("Row-vector × matrix shape mismatch")
        return RowVector(*(sum(x * y for x, y in zip(other.values, column))
                           for column in zip(*self.rows)))

### Step 2 — Compute one rectangular example by hand

Take \(A = [[1,2,3],[4,5,6]]\) and \(B = [[7,8],[9,10],[11,12]]\). The top-left entry of \(AB\) is \(1·7 + 2·9 + 3·11 = 58\). The result must have shape `(2, 2)`. In contrast, multiplying `A` by a 3-component vector produces 2 components.

In [22]:
A = Matrix([[1, 2, 3], [4, 5, 6]])
B = Matrix([[7, 8], [9, 10], [11, 12]])
assert A.shape == (2, 3)
assert A.T.shape == (3, 2)
assert (A @ B).rows == ((58, 64), (139, 154))
assert (A @ RowVector(1, 0, -1)) == RowVector(-2, -2)
assert (RowVector(2, 3) @ A) == RowVector(14, 19, 24)
assert A + A == 2 * A
print("A @ B =", A @ B)
print("row @ A =", RowVector(2, 3) @ A)

A @ B = Matrix(rows=((58, 64), (139, 154)))
row @ A = RowVector(values=(14, 19, 24))


### Step 3 — Investigate shape failures and the meaning of `*` versus `@`

`A * B` is deliberately undefined: matrix multiplication is explicitly spelled `A @ B`. This prevents ambiguity with elementwise multiplication. The `@` operator follows inner-dimension compatibility, whereas `+` requires exactly matching shapes.

In [23]:
expect_error(TypeError, lambda: A * B)
expect_error(ValueError, lambda: A + B)
expect_error(ValueError, lambda: A @ A)
expect_error(ValueError, lambda: A @ RowVector(1, 2))
expect_error(ValueError, lambda: RowVector(1, 2, 3) @ A)
expect_error(ValueError, lambda: Matrix([[1, 2], [3]]))

Expected TypeError: unsupported operand type(s) for *: 'Matrix' and 'Matrix'
Expected ValueError: Cannot add shapes (2, 3) and (3, 2)
Expected ValueError: Cannot multiply shapes (2, 3) and (2, 3)
Expected ValueError: Matrix × column-vector shape mismatch
Expected ValueError: Row-vector × matrix shape mismatch
Expected ValueError: Expected a nonempty rectangular matrix


### Step 4 — Verify the transpose identity

For compatible matrices, \((AB)^T=B^TA^T\). Testing a structural identity catches more mistakes than a single expected result: it exercises both matrix multiplication and transpose with different shapes.

In [24]:
assert (A @ B).T == B.T @ A.T
identity_3 = Matrix([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
assert A @ identity_3 == A
print("Transpose and identity laws: passed")

Transpose and identity laws: passed


**Extension to try:** Implement `Matrix.__neg__` and matrix subtraction, then verify \(A-(B+C)=(A-B)-C\) for same-shaped matrices. For non-integer floating-point inputs, compare entries with tolerances.

---
# Problem 7 — Arithmetic in a modular ring, including division

**Difficulty:** Domain-dependent arithmetic and reflected operators.

Create `ModInt(value, modulus)` supporting addition, subtraction, multiplication, integer powers, and division when an inverse exists. Values are canonicalized into `0 ... modulus-1`. A plain integer acts as an element of the **same modulus**, but mixing two `ModInt` values with different moduli is an error.

Consider the difference between modulus 7 (prime) and modulus 8 (composite): in modulus 8, the element 2 has **no multiplicative inverse**. Division cannot always succeed.

### Step 1 — Normalize representation and operand types

Use a frozen dataclass to store the reduced residue. Reject bool even though `isinstance(True, int)` is true. The conversion method will distinguish an unsupported operand type from an invalid pair of supported modular values.

In [25]:
@dataclass(frozen=True)
class ModInt:
    value: int
    modulus: int

    def __post_init__(self):
        if type(self.value) is not int or type(self.modulus) is not int:
            raise TypeError("Value and modulus must be integers, not bool")
        if self.modulus < 2:
            raise ValueError("Modulus must be at least 2")
        object.__setattr__(self, "value", self.value % self.modulus)

    def _coerce(self, other):
        if isinstance(other, ModInt):
            if other.modulus != self.modulus:
                raise ValueError("Cannot mix different moduli")
            return other
        if type(other) is int:
            return ModInt(other, self.modulus)
        return NotImplemented

    def __add__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return ModInt(self.value + other.value, self.modulus)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return ModInt(-self.value, self.modulus)

    def __sub__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return self + (-other)

    def __rsub__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return other - self

    def __mul__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return ModInt(self.value * other.value, self.modulus)

    def __rmul__(self, other):
        return self * other

    def inverse(self):
        try:
            return ModInt(pow(self.value, -1, self.modulus), self.modulus)
        except ValueError as exc:
            raise ZeroDivisionError("Element is not invertible modulo the modulus") from exc

    def __truediv__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return self * other.inverse()

    def __rtruediv__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return other / self

    def __pow__(self, exponent):
        if type(exponent) is not int:
            raise TypeError("Exponent must be an integer")
        if exponent >= 0:
            return ModInt(pow(self.value, exponent, self.modulus), self.modulus)
        inverse = self.inverse()
        return ModInt(pow(inverse.value, -exponent, self.modulus), self.modulus)

### Step 2 — Walk through multiplication, division, and negative powers

Modulo 7, the inverse of 3 is 5 because \(3·5=15\equiv1\pmod7\). Thus \(2/3\equiv2·5\equiv3\pmod7\). The negative power `3 ** -1` means the inverse **inside this type**, not Python's normal floating-point reciprocal of an `int`.

In [26]:
u = ModInt(2, 7)
v = ModInt(3, 7)
assert ModInt(16, 7) == ModInt(2, 7)
assert u + v == ModInt(5, 7)
assert 10 - u == ModInt(1, 7)
assert u * v == ModInt(6, 7)
assert v.inverse() == ModInt(5, 7)
assert u / v == ModInt(3, 7)
assert 1 / v == ModInt(5, 7)
assert v ** -1 == ModInt(5, 7)
assert v ** 100 == ModInt(pow(3, 100, 7), 7)
print("2 / 3 (mod 7) =", u / v)

2 / 3 (mod 7) = ModInt(value=3, modulus=7)


### Step 3 — Test the cases that divide a ring from a field

Not every nonzero residue is invertible under a composite modulus: e.g. `ModInt(2, 8)`. We raise a clear division error rather than returning a misleading numeric value. Notice also that `ModInt(3, 7) + ModInt(3, 8)` must not silently pick either modulus.

In [27]:
expect_error(ZeroDivisionError, lambda: ModInt(1, 8) / ModInt(2, 8))
expect_error(ZeroDivisionError, lambda: ModInt(2, 8) ** -1)
expect_error(ValueError, lambda: ModInt(3, 7) + ModInt(3, 8))
expect_error(TypeError, lambda: ModInt(3, 7) + 1.5)
expect_error(TypeError, lambda: ModInt(True, 7))
assert ModInt(3, 8) * ModInt(3, 8).inverse() == ModInt(1, 8)

Expected ZeroDivisionError: Element is not invertible modulo the modulus
Expected ZeroDivisionError: Element is not invertible modulo the modulus
Expected ValueError: Cannot mix different moduli
Expected TypeError: unsupported operand type(s) for +: 'ModInt' and 'float'
Expected TypeError: Value and modulus must be integers, not bool


**Extension to try:** Implement a method that lists all invertible residues for a given modulus. Check your output against the mathematical criterion `gcd(value, modulus) == 1` (available in `math`).

---
# Problem 8 — Decimal money, rounding policies, and rejected operations

**Difficulty:** Arithmetic correctness in a business domain.

Create a `Money` type with a `Decimal` amount and a three-letter currency code. Amounts are quantized to cents using **round-half-to-even**. Support same-currency addition/subtraction, multiplication by an exact integer or Decimal scalar, division by a scalar, and same-currency `Money / Money` producing a **dimensionless Decimal ratio**.

**Important boundary:** This is a pedagogical two-decimal-currency model. Real currencies can use different minor units, cash-rounding rules, and legally defined settlement conventions. Never use a bare `float` for values here; accept decimal strings, integers, or `Decimal` instead.

### Step 1 — Decide when rounding happens

We round at **construction of every Money value**. This is easy to state and test, but means intermediate arithmetic can matter. The exchange operation will not be overloaded as `+`: currency conversion requires an explicit rate, time, and source. In this tutorial, different-currency addition is rejected rather than guessed.

In [28]:
CENT = Decimal("0.01")


def money_decimal(value):
    if isinstance(value, bool) or isinstance(value, float):
        raise TypeError("Use Decimal, integer, or decimal string; floats are rejected")
    if not isinstance(value, (str, int, Decimal)):
        raise TypeError("Unsupported monetary amount or scalar")
    result = Decimal(value)
    if not result.is_finite():
        raise ValueError("Monetary values must be finite")
    return result

assert money_decimal("1.25") == Decimal("1.25")
expect_error(TypeError, lambda: money_decimal(0.1))

Expected TypeError: Use Decimal, integer, or decimal string; floats are rejected


### Step 2 — Write the class with strict currency boundaries

Unsupported operand types return `NotImplemented`, allowing Python's arithmetic protocol to do its work. A recognized `Money` object with the wrong currency raises `ValueError`. Multiplication by another Money is **not** a price: it creates squared currency units and is intentionally disallowed. Division by another Money of the same currency is different: its result is a pure ratio.

In [29]:
@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def __post_init__(self):
        if (not isinstance(self.currency, str) or len(self.currency) != 3
                or not self.currency.isascii() or not self.currency.isalpha()
                or self.currency != self.currency.upper()):
            raise ValueError("Use a three-letter uppercase ASCII currency code")
        amount = money_decimal(self.amount)
        object.__setattr__(self, "amount", amount.quantize(CENT, rounding=ROUND_HALF_EVEN))

    def _require_same_currency(self, other):
        if self.currency != other.currency:
            raise ValueError(f"Different currencies: {self.currency}, {other.currency}")

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._require_same_currency(other)
        return Money(self.amount + other.amount, self.currency)

    def __neg__(self):
        return Money(-self.amount, self.currency)

    def __sub__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self + (-other)

    def __mul__(self, scalar):
        if isinstance(scalar, Money) or isinstance(scalar, float) or isinstance(scalar, bool):
            return NotImplemented
        if not isinstance(scalar, (int, Decimal)):
            return NotImplemented
        return Money(self.amount * money_decimal(scalar), self.currency)

    def __rmul__(self, scalar):
        return self * scalar

    def __truediv__(self, other):
        if isinstance(other, Money):
            self._require_same_currency(other)
            return self.amount / other.amount
        if isinstance(other, (int, Decimal)) and not isinstance(other, bool):
            return Money(self.amount / money_decimal(other), self.currency)
        return NotImplemented

### Step 3 — Inspect the result **types**, not just printed values

A price divided by a count is still Money; a price divided by another same-currency price is a dimensionless Decimal. This is a good example of how operator overloads can return *different types* depending on the other operand, provided the contract is explicit.

In [30]:
price = Money("19.995", "USD")
fee = Money("0.50", "USD")
assert price.amount == Decimal("20.00")         # half-to-even
assert price + fee == Money("20.50", "USD")
assert price - fee == Money("19.50", "USD")
assert price * 3 == Money("60.00", "USD")
assert Decimal("0.25") * price == Money("5.00", "USD")
assert price / 4 == Money("5.00", "USD")
assert price / Money("5.00", "USD") == Decimal("4")
print("Rounded price:", price, "| price / price:", price / price)

Rounded price: Money(amount=Decimal('20.00'), currency='USD') | price / price: 1


### Step 4 — Expose an intermediate-rounding surprise

Take a tiny amount that rounds to two cents. Applying a tax rate *after* rounding need not equal applying that rate to the original unrounded input and rounding once. We must make rounding policy visible instead of implying arithmetic is exact at every intermediate step.

In [31]:
already_rounded = Money("0.015", "USD")
then_scaled = already_rounded * Decimal("1.5")
scaled_first = Money(Decimal("0.015") * Decimal("1.5"), "USD")
assert already_rounded.amount == Decimal("0.02")
assert then_scaled.amount == Decimal("0.03")
assert scaled_first.amount == Decimal("0.02")
assert then_scaled != scaled_first
print("round then scale:", then_scaled, "| scale then round:", scaled_first)

round then scale: Money(amount=Decimal('0.03'), currency='USD') | scale then round: Money(amount=Decimal('0.02'), currency='USD')


### Step 5 — Test the forbidden operations explicitly

Reflected addition of `0 + Money` is not implemented, so the default `sum([money_1, money_2])` fails on its integer starting value. Rather than silently blessing all integer/money additions, pass an explicit zero of the right currency to `sum`.

In [32]:
assert sum([price, fee], Money("0", "USD")) == Money("20.50", "USD")
expect_error(TypeError, lambda: sum([price, fee]))
expect_error(ValueError, lambda: price + Money("1", "EUR"))
expect_error(ValueError, lambda: price / Money("1", "EUR"))
expect_error(TypeError, lambda: price * Money("2", "USD"))
expect_error(TypeError, lambda: price * 0.1)
expect_error(ZeroDivisionError, lambda: price / 0)
expect_error(ValueError, lambda: Money("Infinity", "USD"))

Expected TypeError: unsupported operand type(s) for +: 'int' and 'Money'
Expected ValueError: Different currencies: USD, EUR
Expected ValueError: Different currencies: USD, EUR
Expected TypeError: unsupported operand type(s) for *: 'Money' and 'Money'
Expected TypeError: unsupported operand type(s) for *: 'Money' and 'float'
Expected DivisionByZero: [<class 'decimal.DivisionByZero'>]
Expected ValueError: Monetary values must be finite


**Extension to try:** Add an explicit `convert(rate: Decimal, target_currency: str)` method. Document precisely whether `rate` means target-units-per-source-unit or the reverse, and apply the exchange rate **before** the destination-currency rounding step.

---
# Problem 9 — Exact interval arithmetic and a subtle dependency trap

**Difficulty:** Rigorous operand order and numerical semantics.

An interval `[lo, hi]` represents every number between its endpoints. Use exact `Fraction` endpoints and implement addition, subtraction, multiplication, division, reversed subtraction/division, absolute value, and nonnegative integer powers. Accept integer or `Fraction` scalars as degenerate intervals, but reject floating-point values: converting an inexact float to an exact fraction can confuse the distinction between decimal input and its binary approximation.

Division is undefined if the denominator interval **contains zero**. Powers need special care: squaring an interval that crosses zero has lower bound **zero**, not the minimum of its endpoint squares.

### Step 1 — Derive each bound carefully

For \(X=[a,b]\), \(Y=[c,d]\):

- `X + Y = [a+c, b+d]`.
- `X - Y = [a-d, b-c]`: note the reversed endpoints on the right.
- `X * Y` takes the min and max of the **four endpoint products**.
- If `Y` excludes zero, `1 / Y = [min(1/c,1/d), max(1/c,1/d)]`, then `X / Y = X * (1/Y)`.

These formulas enclose every result under exact real arithmetic. Since `Fraction` is exact for rational inputs, we avoid floating-point rounding concerns for the bounds in this exercise.

In [33]:
def as_fraction(value):
    if type(value) is int or isinstance(value, Fraction):
        return Fraction(value)
    raise TypeError("Intervals accept only integer or Fraction endpoints")

assert as_fraction(2) == Fraction(2)
expect_error(TypeError, lambda: as_fraction(0.1))

Expected TypeError: Intervals accept only integer or Fraction endpoints


### Step 2 — Implement the full interval API

All operations build a new interval. `_coerce` turns a supported scalar into `[scalar, scalar]`, so reflected subtraction and division can use the correct operand order rather than naively forwarding the operation. `abs(X)` returns an **interval of possible magnitudes**, not a single number.

In [34]:
@dataclass(frozen=True)
class Interval:
    lo: Fraction
    hi: Fraction

    def __post_init__(self):
        lo, hi = as_fraction(self.lo), as_fraction(self.hi)
        if lo > hi:
            raise ValueError("Interval lower endpoint exceeds upper endpoint")
        object.__setattr__(self, "lo", lo)
        object.__setattr__(self, "hi", hi)

    @staticmethod
    def _coerce(other):
        if isinstance(other, Interval):
            return other
        if type(other) is int or isinstance(other, Fraction):
            return Interval(other, other)
        return NotImplemented

    def __add__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return Interval(self.lo + other.lo, self.hi + other.hi)

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return Interval(-self.hi, -self.lo)

    def __sub__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return Interval(self.lo - other.hi, self.hi - other.lo)

    def __rsub__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return other - self

    def __mul__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        products = (self.lo * other.lo, self.lo * other.hi,
                    self.hi * other.lo, self.hi * other.hi)
        return Interval(min(products), max(products))

    def __rmul__(self, other):
        return self * other

    def reciprocal(self):
        if self.lo <= 0 <= self.hi:
            raise ZeroDivisionError("Denominator interval contains zero")
        return Interval(min(1 / self.lo, 1 / self.hi),
                        max(1 / self.lo, 1 / self.hi))

    def __truediv__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return self * other.reciprocal()

    def __rtruediv__(self, other):
        other = self._coerce(other)
        if other is NotImplemented:
            return NotImplemented
        return other / self

    def __abs__(self):
        if self.lo >= 0:
            return self
        if self.hi <= 0:
            return -self
        return Interval(0, max(-self.lo, self.hi))

    def __pow__(self, power):
        if type(power) is not int or power < 0:
            raise ValueError("This interval type supports nonnegative integer powers")
        if power == 0:
            return Interval(1, 1)
        if power % 2 == 1:
            return Interval(self.lo ** power, self.hi ** power)
        if self.lo <= 0 <= self.hi:
            return Interval(0, max(self.lo ** power, self.hi ** power))
        return Interval(min(self.lo ** power, self.hi ** power),
                        max(self.lo ** power, self.hi ** power))

### Step 3 — Compute positive, negative, and zero-crossing cases

Subtraction and division expose why reflected methods matter: `10 - [2, 4]` is `[6, 8]`, whereas `[2, 4] - 10` is `[-8, -6]`. Squaring `[-2, 3]` yields `[0, 9]`, not `[4, 9]`.

In [35]:
X, Y = Interval(2, 4), Interval(-1, 3)
assert X + Y == Interval(1, 7)
assert X - Y == Interval(-1, 5)
assert 10 - X == Interval(6, 8)
assert X - 10 == Interval(-8, -6)
assert X * Y == Interval(-4, 12)
assert 2 / X == Interval(Fraction(1, 2), 1)
assert abs(Interval(-2, 3)) == Interval(0, 3)
assert Interval(-2, 3) ** 2 == Interval(0, 9)
assert Interval(-2, -1) ** 2 == Interval(1, 4)
assert Interval(-2, 3) ** 3 == Interval(-8, 27)
print("Bounds for X*Y:", X * Y)

Bounds for X*Y: Interval(lo=Fraction(-4, 1), hi=Fraction(12, 1))


### Step 4 — Expose the dependency problem

Suppose `X` denotes one unknown number in `[2,4]`. The mathematical expression `x - x` is always zero, but interval evaluation `X - X` returns `[-2,2]`. Each occurrence is treated independently during ordinary interval arithmetic, so the result can be **conservative but wider than necessary**. This is not a bug in our subtraction formula.

In [36]:
uncertain = Interval(2, 4)
assert uncertain - uncertain == Interval(-2, 2)
assert (uncertain - uncertain) != Interval(0, 0)
expect_error(ZeroDivisionError, lambda: X / Interval(-1, 1))
expect_error(ValueError, lambda: Interval(5, 2))
expect_error(TypeError, lambda: X + 0.1)
print("Dependency demonstration:", uncertain - uncertain)

Expected ZeroDivisionError: Denominator interval contains zero
Expected ValueError: Interval lower endpoint exceeds upper endpoint
Expected TypeError: unsupported operand type(s) for +: 'Interval' and 'float'
Dependency demonstration: Interval(lo=Fraction(-2, 1), hi=Fraction(2, 1))


**Extension to try:** Design a `contains(value)` method and test whether the results of randomly sampled rational operand pairs always fall inside the computed interval. Be sure to skip divisors that contain zero.

---
# Problem 10 — Build a symbolic expression tree with operator overloading

**Difficulty:** Immutable syntax trees and recursion.

Allow expressions such as `x*x + 3*x + 2` to create a tree instead of immediately producing a number. Implement addition, subtraction, multiplication, unary negation, nonnegative integer powers, and reflected scalar operations. Then implement `evaluate(environment)` and `differentiate(variable)` using symbolic differentiation rules.

Here the overloaded operators **construct an expression**, a very different meaning from vector or money arithmetic. This is legitimate because the representation's purpose is explicit and consistent.

### Step 1 — Choose a grammar and differentiation rules

We will have `Constant`, `Variable`, `AddExpr`, `MulExpr`, `NegExpr`, and `PowExpr`. The base `Expr` implements operator syntax once; every node supplies evaluation and differentiation. Basic rules:

\(\frac{d}{dx}(f+g)=f'+g'\), \(\frac{d}{dx}(fg)=f'g+fg'\), and \(\frac{d}{dx}(f^n)=nf^{n-1}f'\) for nonnegative integer \(n\). A constant differentiates to zero; a variable differentiates to one with respect to itself and zero with respect to any other variable.

For brevity the output is **not automatically simplified**; an expression like `0*x + 1*y` may remain in the tree. This does not change its evaluated value.

In [37]:
class Expr:
    def evaluate(self, env):
        raise NotImplementedError

    def differentiate(self, variable):
        raise NotImplementedError

    def __add__(self, other):
        other = as_expr(other)
        return AddExpr(self, other) if other is not NotImplemented else NotImplemented

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return NegExpr(self)

    def __sub__(self, other):
        other = as_expr(other)
        return self + (-other) if other is not NotImplemented else NotImplemented

    def __rsub__(self, other):
        other = as_expr(other)
        return other - self if other is not NotImplemented else NotImplemented

    def __mul__(self, other):
        other = as_expr(other)
        return MulExpr(self, other) if other is not NotImplemented else NotImplemented

    def __rmul__(self, other):
        return self * other

    def __pow__(self, exponent):
        if type(exponent) is not int or exponent < 0:
            raise ValueError("Powers must be nonnegative integers")
        return PowExpr(self, exponent)


def as_expr(value):
    if isinstance(value, Expr):
        return value
    if isinstance(value, Real) and not isinstance(value, bool) and isfinite(value):
        return Constant(value)
    return NotImplemented

### Step 2 — Implement each node with one clear responsibility

The nodes below are immutable records. Operator composition occurs in the base class; specific node behavior occurs in `evaluate` and `differentiate`. This avoids writing six arithmetic methods for every node type.

In [38]:
@dataclass(frozen=True)
class Constant(Expr):
    value: Real

    def evaluate(self, env):
        return self.value

    def differentiate(self, variable):
        return Constant(0)


@dataclass(frozen=True)
class Variable(Expr):
    name: str

    def evaluate(self, env):
        return env[self.name]

    def differentiate(self, variable):
        return Constant(1 if self.name == variable else 0)


@dataclass(frozen=True)
class AddExpr(Expr):
    left: Expr
    right: Expr

    def evaluate(self, env):
        return self.left.evaluate(env) + self.right.evaluate(env)

    def differentiate(self, variable):
        return self.left.differentiate(variable) + self.right.differentiate(variable)


@dataclass(frozen=True)
class MulExpr(Expr):
    left: Expr
    right: Expr

    def evaluate(self, env):
        return self.left.evaluate(env) * self.right.evaluate(env)

    def differentiate(self, variable):
        return (self.left.differentiate(variable) * self.right +
                self.left * self.right.differentiate(variable))


@dataclass(frozen=True)
class NegExpr(Expr):
    argument: Expr

    def evaluate(self, env):
        return -self.argument.evaluate(env)

    def differentiate(self, variable):
        return -self.argument.differentiate(variable)


@dataclass(frozen=True)
class PowExpr(Expr):
    base: Expr
    exponent: int

    def evaluate(self, env):
        return self.base.evaluate(env) ** self.exponent

    def differentiate(self, variable):
        if self.exponent == 0:
            return Constant(0)
        return self.exponent * (self.base ** (self.exponent - 1)) * self.base.differentiate(variable)

### Step 3 — Construct an expression using familiar Python notation

`x*x + 3*x + 2` becomes a tree. Differentiation generates another tree, which we evaluate numerically. This is easier to validate initially than comparing the tree against a particular printed format, because equivalent symbolic expressions need not have identical shapes.

In [39]:
x = Variable("x")
y = Variable("y")
expression = x * x + 3 * x + 2
derivative = expression.differentiate("x")
assert expression.evaluate({"x": 4}) == 30
assert derivative.evaluate({"x": 4}) == 11
assert (7 - x).evaluate({"x": 2}) == 5
assert (x ** 3).differentiate("x").evaluate({"x": 2}) == 12
assert (x * y).differentiate("y").evaluate({"x": 3, "y": 5}) == 3
assert (x ** 0).differentiate("x").evaluate({"x": 8}) == 0
print("Expression at x=4:", expression.evaluate({"x": 4}))
print("Derivative at x=4:", derivative.evaluate({"x": 4}))

Expression at x=4: 30
Derivative at x=4: 11


### Step 4 — Verify a derivative using independent finite differences

A simple numerical check approximates \(f'(x)\) by \([f(x+h)-f(x-h)]/(2h)\). We do not use it as an exact equality because floating-point differences introduce approximation and cancellation errors. It is useful as an independent smoke test of the recursive rules.

In [40]:
def centered_difference(expression, at, h=1e-5):
    return ((expression.evaluate({"x": at + h}) -
             expression.evaluate({"x": at - h})) / (2 * h))

for at in (-3.0, -0.5, 0.0, 2.0, 10.0):
    analytic = derivative.evaluate({"x": at})
    numerical = centered_difference(expression, at)
    assert isclose(analytic, numerical, rel_tol=1e-8, abs_tol=1e-7)
expect_error(TypeError, lambda: x + "unsupported")
expect_error(ValueError, lambda: x ** -2)
expect_error(KeyError, lambda: expression.evaluate({}))
print("Symbolic / finite-difference checks: passed")

Expected TypeError: unsupported operand type(s) for +: 'Variable' and 'str'
Expected ValueError: Powers must be nonnegative integers
Expected KeyError: 'x'
Symbolic / finite-difference checks: passed


**Extension to try:** Add `__truediv__` and a `DivExpr` node using the quotient rule. Define how division by zero is handled at evaluation time. Only add operator syntax once you can describe its evaluation and differentiation semantics precisely.

---
# Problem 11 — Transactional `+=` with mutable inventory

**Difficulty:** Aliasing, invariants, and in-place failure safety.

Create an `Inventory` of item counts keyed by SKU. Adding inventories with `+` returns a **new** object; applying `+=` updates the **same** inventory object. Subtracting inventory removes stock but may not create negative counts. Crucially, a failed `+=` or `-=` must leave the original inventory unchanged.

This contrasts with our immutable polynomial and point designs. In-place mutation can be appropriate when identity represents a shared long-lived collection—but the implementation needs a deliberate transaction boundary.

### Step 1 — Validate every incoming count before making any changes

Counts must be nonnegative integers, excluding booleans; SKU keys must be nonempty strings. Zero counts are removed from the canonical internal dictionary. Exposing a **copy** from `counts` keeps external code from bypassing validation.

In [41]:
class Inventory:
    def __init__(self, counts=None):
        self._counts = self._validated({} if counts is None else counts)

    @staticmethod
    def _validated(counts):
        result = {}
        for sku, count in dict(counts).items():
            if not isinstance(sku, str) or not sku:
                raise TypeError("SKU must be a nonempty string")
            if type(count) is not int:
                raise TypeError("Counts must be integers, not bool")
            if count < 0:
                raise ValueError("Counts cannot be negative")
            if count:
                result[sku] = count
        return result

    @property
    def counts(self):
        return dict(self._counts)

    def __repr__(self):
        return f"Inventory({self._counts!r})"

    def __eq__(self, other):
        if not isinstance(other, Inventory):
            return NotImplemented
        return self._counts == other._counts

    def _combine(self, other, sign):
        if not isinstance(other, Inventory):
            return NotImplemented
        proposed = self.counts
        for sku, count in other._counts.items():
            proposed[sku] = proposed.get(sku, 0) + sign * count
            if proposed[sku] < 0:
                raise ValueError(f"Insufficient inventory for {sku}")
        return Inventory(proposed)

    def __add__(self, other):
        return self._combine(other, 1)

    def __sub__(self, other):
        return self._combine(other, -1)

    def __iadd__(self, other):
        proposed = self + other           # construct and validate first
        if proposed is NotImplemented:
            return NotImplemented
        self._counts = proposed._counts    # one commit after all checks pass
        return self

    def __isub__(self, other):
        proposed = self - other
        if proposed is NotImplemented:
            return NotImplemented
        self._counts = proposed._counts
        return self

### Step 2 — Distinguish a new value from a mutation

`warehouse + delivery` leaves the original unchanged. `warehouse += delivery` changes the original object: every alias sees the update. This is the opposite of our immutable value objects, and it is intentional.

In [42]:
warehouse = Inventory({"A": 5, "B": 2})
delivery = Inventory({"A": 3, "C": 4})
preview = warehouse + delivery
assert preview == Inventory({"A": 8, "B": 2, "C": 4})
assert warehouse == Inventory({"A": 5, "B": 2})
reference = warehouse
warehouse += delivery
assert warehouse is reference
assert reference == preview
assert warehouse.counts == {"A": 8, "B": 2, "C": 4}
print("Alias sees committed change:", reference)

Alias sees committed change: Inventory({'A': 8, 'B': 2, 'C': 4})


### Step 3 — Test rollback and the public-copy boundary

Imagine a removal involving two SKUs: if the second fails, the first must **not** have been removed. We avoid partial updates by computing and validating `proposed` before the single assignment to the original object's dictionary.

In [43]:
snapshot = warehouse.counts
expect_error(ValueError, lambda: warehouse.__isub__(Inventory({"A": 1, "B": 99})))
assert warehouse.counts == snapshot
stock_copy = warehouse.counts
stock_copy["A"] = 100000
assert warehouse.counts == snapshot
warehouse -= Inventory({"A": 8, "C": 4})
assert warehouse.counts == {"B": 2}
expect_error(TypeError, lambda: Inventory({"A": True}))
expect_error(ValueError, lambda: Inventory({"A": -1}))
expect_error(TypeError, lambda: warehouse + 4)
print("Rollback and defensive-copy tests: passed")

Expected ValueError: Insufficient inventory for B
Expected TypeError: Counts must be integers, not bool
Expected ValueError: Counts cannot be negative
Expected TypeError: unsupported operand type(s) for +: 'Inventory' and 'int'
Rollback and defensive-copy tests: passed


**Important qualification:** This example provides exception-safe, whole-operation updates in ordinary sequential execution; it is **not** a thread-safe or database transaction. Concurrent writers would require synchronization and often persistence-level guarantees.

**Extension to try:** Add a `reserve(request)` operation returning an independent reservation object. Think through what happens if a later cancellation fails.

---
# Problem 12 — Design randomized tests for your arithmetic contracts

**Difficulty:** Testing strategy and algebraic reasoning.

Handpicked assertions check familiar examples; randomized tests explore many less obvious combinations. Write deterministic tests with a fixed seed so failures are reproducible. Choose identities **only when they truly apply**: exact integer polynomial arithmetic obeys distributivity, but rounded money arithmetic can be sensitive to where rounding occurs, and interval arithmetic may over-approximate due to dependency.

### Step 1 — Generate reproducible polynomial cases

Check commutativity and distributivity for short polynomials with integer coefficients. If one test fails, print the seed and input objects so you can reproduce it without guessing. We include the zero polynomial in the generator to probe degree and normalization boundaries.

In [44]:
SEED = 20260919
rng = Random(SEED)


def random_polynomial():
    length = rng.randint(1, 5)
    return Polynomial(*(rng.randint(-5, 5) for _ in range(length)))


for test_index in range(200):
    a, b, c = random_polynomial(), random_polynomial(), random_polynomial()
    assert a + b == b + a, (SEED, test_index, a, b)
    assert a * (b + c) == a * b + a * c, (SEED, test_index, a, b, c)
    assert a - a == Polynomial(0), (SEED, test_index, a)
print("Passed 200 deterministic polynomial property checks; seed:", SEED)

Passed 200 deterministic polynomial property checks; seed: 20260919


### Step 2 — Independently check sparse vector operations against a dense reference

An excellent test uses a **different implementation** as its oracle. Here sparse dot products are compared with a simple dense loop over all indices for a small dimension. This makes it less likely that the same mistaken algorithm appears in both the implementation and its test.

In [45]:
for trial in range(100):
    dimension = rng.randint(1, 20)
    dense_left = [rng.randint(-3, 3) for _ in range(dimension)]
    dense_right = [rng.randint(-3, 3) for _ in range(dimension)]
    left = SparseVector(dimension, enumerate(dense_left))
    right = SparseVector(dimension, enumerate(dense_right))
    expected_dot = sum(x * y for x, y in zip(dense_left, dense_right))
    assert left @ right == expected_dot, (trial, left, right)
    expected_sum = {i: x + y for i, (x, y) in
                    enumerate(zip(dense_left, dense_right)) if x + y}
    assert (left + right).as_dict() == expected_sum
print("Passed 100 sparse/dense cross-checks")

Passed 100 sparse/dense cross-checks


### Step 3 — Sample actual values to test an interval enclosure

Generate short integer intervals and enumerate all integer points inside them. Every concrete arithmetic result must lie inside the corresponding interval result. This is a finite test, not a proof for every rational or real input, but it is good at catching sign and endpoint mistakes.

In [46]:
def contains(interval, value):
    return interval.lo <= value <= interval.hi


for trial in range(100):
    endpoints_x = sorted((rng.randint(-5, 5), rng.randint(-5, 5)))
    endpoints_y = sorted((rng.randint(-5, 5), rng.randint(-5, 5)))
    left = Interval(*endpoints_x)
    right = Interval(*endpoints_y)
    for x_value in range(int(left.lo), int(left.hi) + 1):
        for y_value in range(int(right.lo), int(right.hi) + 1):
            assert contains(left + right, x_value + y_value)
            assert contains(left - right, x_value - y_value)
            assert contains(left * right, x_value * y_value)
print("Passed 100 randomized interval enclosure checks")

Passed 100 randomized interval enclosure checks


### Step 4 — A final testing checklist

When designing a new operator overload, ask:

1. **Domain:** Which types and combinations make mathematical sense? Which ones return `NotImplemented`?
2. **Invariants:** What must always hold after construction and after every operation?
3. **Order:** Are `a-b`, `b-a`, `a/b`, and `b/a` intentionally different?
4. **Identity:** Does `+=` mutate, or return a new object and rebind the name? What happens to aliases?
5. **Errors:** Which invalid inputs raise `TypeError`, `ValueError`, or `ZeroDivisionError`? Can a failed in-place operation leave partial state?
6. **Precision:** Are values exact, floating-point approximations, rounded decimals, or bounds on uncertainty?
7. **Evidence:** Do you have edge-case tests, algebraic properties where valid, and at least one independent reference implementation?

Do not assume that an operator is useful merely because Python lets you overload it. The goal is a predictable interface whose behavior readers can discover from the type's purpose.

In [47]:
print("ALL TUTORIAL PROBLEMS COMPLETED SUCCESSFULLY")
print("Covered 12 problems across dispatch, polynomial algebra, units, geometry,")
print("sparse vectors, matrix algebra, modular numbers, money, intervals,")
print("symbolic expressions, mutation safety, and randomized testing.")

ALL TUTORIAL PROBLEMS COMPLETED SUCCESSFULLY
Covered 12 problems across dispatch, polynomial algebra, units, geometry,
sparse vectors, matrix algebra, modular numbers, money, intervals,
symbolic expressions, mutation safety, and randomized testing.
